# IQ GLM-5.3 — Fresh Google Colab setup
**Run cells 1–6 in order in a newly connected Colab runtime.** Previous version wrongly expected checkpoint mounts, an IQ checkout and other prerequisites.

This notebook installs only the Hugging Face bucket SDK, detects the *actually attached* CPU/GPU/TPU, tests tiny synthetic tensors, and connects directly to `kofecheckmate/iq-glm53-all256`. It only reads ~6.9 MB of JSON metadata, **never 539 GB of model weights**.

**Before running:** For a private bucket, create a read-permission [Hugging Face token](https://huggingface.co/settings/tokens). In Colab's left **Secrets** panel (key icon), add `HF_TOKEN` and toggle **Notebook access**. Do not paste tokens into the notebook, GitHub or this chat. This authorizes *your IQ bucket only*; Meta/Llama permission is not needed for these checks.

References: [HF bucket access](https://huggingface.co/docs/hub/storage-buckets-access) · [HF filesystem](https://huggingface.co/docs/huggingface_hub/package_reference/hf_file_system) · [Colab hardware](https://research.google.com/colaboratory/faq.html).


In [ ]:
# 1 — Install only fresh-session prerequisites. No model downloads.
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "huggingface_hub>=1.5.0", "fsspec"])
from huggingface_hub import HfFileSystem, HfApi
print("HF Storage Bucket filesystem ready.")


In [ ]:
# 2 — Retrieve private-bucket credentials securely. No printing or saving tokens.
import os
hf_token = os.environ.get("HF_TOKEN") or None
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if not hf_token:
    print("HF_TOKEN missing: add it in Colab Secrets, enable Notebook access, rerun this cell.")
else:
    try:
        who = HfApi(token=hf_token).whoami()
        print("HF authenticated:", who.get("name", "(unknown)"))
    except Exception as e:
        print("HF authentication failed:", type(e).__name__, str(e)[:250])
        hf_token = None


In [ ]:
# 3 — Detect actual TPU/JAX, NVIDIA GPU/CUDA, or CPU and run a tiny synthetic smoke test.
# Downloads a SMALL Python probe script, not model files. Does not provision GPUs/TPUs.
import json, subprocess, sys
from pathlib import Path
from urllib.request import urlopen
source=("https://raw.githubusercontent.com/Harqer/IQ/"
        "fix/iq-full256-mapped-inference-preflight-20261114/scripts/iq_colab_runtime_probe.py")
probe=Path("/tmp/iq_colab_runtime_probe.py")
with urlopen(source, timeout=30) as response:
    probe.write_bytes(response.read())
result=subprocess.run([sys.executable,str(probe),"--smoke"],text=True,capture_output=True)
print(result.stdout or result.stderr)
if result.returncode:
    print("Probe requires attention: exit code", result.returncode)
try:
    hardware_report=json.loads(result.stdout)
except Exception:
    hardware_report={"runtime_backend":"undetermined","error":result.stderr[-350:]}


In [ ]:
# 4 — Read-only verification DIRECTLY from your HF bucket; no Colab mount needed.
import hashlib, json, re
from collections import defaultdict
bucket="hf://buckets/kofecheckmate/iq-glm53-all256"
run=bucket+"/IQ/runs/glm53-full78-local-20261025a"
map_path=bucket+"/IQ/device-maps/full256-hybrid-pattern-h200x8.metadata-plan.json"
metadata={"passed":False,"ready_for_full_inference":False,"weights_downloaded":False}
if not hf_token:
    metadata["blocked"]="HF_TOKEN missing or invalid; see cell 2."
else:
    try:
        fs=HfFileSystem(token=hf_token)
        def getjson(uri,maximum):
            with fs.open(uri,"rb") as f:
                b=f.read(maximum+1)
            if len(b)>maximum:
                raise ValueError("Metadata exceeded size cap: "+uri)
            return json.loads(b),b
        cfg,cb=getjson(run+"/iq_config.json",100_000)
        idx,ib=getjson(run+"/model.safetensors.index.json",10_000_000)
        status,_=getjson(run+"/COMPILE_COMPLETE.json",30_000)
        plan,pb=getjson(map_path,2_000_000)
        schedule=cfg["schedule"]["layers"]
        weight_map=idx["weight_map"]
        owners=plan["layer_to_gpu"]
        err=[]
        if hashlib.sha256(cb).hexdigest()!=plan["config_sha256"]:err.append("Config SHA-256")
        if hashlib.sha256(ib).hexdigest()!=plan["index_sha256"]:err.append("Index SHA-256")
        if len(schedule)!=235 or len(owners)!=235:err.append("Physical layer count")
        if schedule.count("mamba3")!=79 or schedule.count("moe")!=78:err.append("Mamba/MoE count")
        if (cfg["mamba3"]["mimo_rank"],cfg["mamba3"]["chunk_size"])!=(4,16):err.append("MIMO rank/chunk")
        if cfg["attnres"]["num_layers"]!=len(schedule):err.append("AttnRes length")
        if len(weight_map)!=63164 or len(set(weight_map.values()))!=20283:err.append("Shard/key index count")
        if idx["metadata"]["total_size"]!=plan["total_weight_bytes"]:err.append("Total indexed bytes")
        if len(set(owners))!=8 or any(a>b for a,b in zip(owners,owners[1:])):err.append("GPU map contiguity")
        if plan.get("inference_ready") or not plan.get("plan_only"):err.append("Unverified ready flag")
        groups=defaultdict(set)
        for key,shard in weight_map.items():
            layer=re.match(r"layers\.(\d+)\.",key)
            mixer=re.match(r"attnres\.mixers\.(\d+)\.",key)
            gpu=(owners[int(layer.group(1))] if layer else
                 owners[int(mixer.group(1))] if mixer else
                 0 if key=="embed_tokens.weight" else 7)
            groups[shard].add(gpu)
        if set(groups)!=set(plan["shard_to_gpu"]):err.append("Shard device coverage")
        else:
            bad=0
            for shard,devices in groups.items():
                expected=next(iter(devices)) if len(devices)==1 else sorted(devices)
                bad+=plan["shard_to_gpu"][shard]!=expected
            if bad:err.append(f"{bad} shard-device mismatches")
        metadata.update({"passed":not err,"errors":err,
            "physical_layers":len(schedule),"tensors_indexed":len(weight_map),
            "shards_indexed":len(groups),
            "checkpoint_GB":round(idx["metadata"]["total_size"]/1e9,3),
            "JSON_bytes_read":len(cb)+len(ib)+len(pb),
            "device_map_GPUs":plan["device_count"],
            "original_compile_status":status.get("status")})
    except Exception as e:
        metadata["blocked"]=f"{type(e).__name__}: {str(e)[:450]}"
print(json.dumps(metadata,indent=2))


In [ ]:
# 5 — Memory-capacity verdict, without loading IQ.
capacity={"can_fit_full256":False}
if not metadata.get("passed"):
    capacity["reason"]="HF bucket metadata check did not pass"
else:
    gpu_info=hardware_report.get("nvidia_gpu",{}).get("devices",[])
    kind=hardware_report.get("runtime_backend")
    if kind!="cuda":
        capacity["reason"]="IQ's pinned Mamba-3 MIMO runtime requires CUDA (TPU path checks JAX only)"
    elif len(gpu_info)<plan["device_count"]:
        capacity["reason"]=f"Need {plan['device_count']} CUDA devices; Colab has {len(gpu_info)}"
    else:
        reserve=float(plan.get("memory_reserve_GB",20))
        insuff=[s["gpu"] for s in plan["placements"]
                if gpu_info[s["gpu"]]["memory_bytes"]/1e9 < s["weight_GB"]+reserve]
        capacity.update({"can_fit_full256":not insuff,
                         "reason":("Devices with insufficient nominal VRAM: "+str(insuff))
                                  if insuff else "Nominal capacity only; kernel/loader not verified"})
print(json.dumps(capacity,indent=2))
print("No IQ weights loaded; no IQ inference executed.")


### 6 — Optional: clone IQ **code** to `/content/IQ` (not model weights)
Only needed if you want the repo's CPU regression tests in this Colab session. This step uses `--depth 1` and `GIT_LFS_SKIP_SMUDGE=1` to avoid pulling large weight blobs. Full IQ generation is not attempted.


In [ ]:
RUN_REPO_TESTS=True
from pathlib import Path
import os,sys,subprocess
repo=Path("/content/IQ")
if RUN_REPO_TESTS:
    if not (repo/".git").is_dir():
        env=dict(os.environ,GIT_LFS_SKIP_SMUDGE="1")
        clone=subprocess.run(["git","clone","--depth","1","--filter=blob:none",
            "--branch","fix/iq-full256-mapped-inference-preflight-20261114",
            "https://github.com/Harqer/IQ.git",str(repo)],
            capture_output=True,text=True,env=env,timeout=180)
        if clone.returncode:
            print("Checkout failed:",clone.stderr[-600:])
    if (repo/".git").is_dir():
        for suite in ["test_iq_device_map_preflight.py","test_iq_colab_runtime_probe.py"]:
            test=subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-p",suite],
                cwd=repo,capture_output=True,text=True,timeout=90)
            print(suite,"PASS" if test.returncode==0 else "FAILED",test.stderr[-260:])
print("Done. This notebook has not downloaded checkpoint weights or executed IQ inference.")
